# Check how many operations would occur in iterations

In [24]:
from similarity import load_og_sentences
from tucker_tensor import TuckerDecomposition
from utils import DATA_DIR

import os
vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_english_vectors.csv")

dataset = "fineweb-en"
random_state = 1
tucker = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    method="siiShifted",
    divergence="fr",
    dims=1000,
    rank=150,
    iterations=500
)
sentence_sample = load_og_sentences(vector_path=vector_path
                                             )
len(sentence_sample)

3729954

In [29]:
sentence_sample[:5]

[('award', 'Martin', '~'),
 ('take', 'owner', 'customer'),
 ('spin', 'Casino', 'slot'),
 ('require', 'journey', 'collaboration'),
 ('leave', 'identity', 'impression')]

In [25]:
tucker.vocab.keys()
for vocab_key in ("vocab_s", "vocab_o"):
    vocab = tucker.vocab[vocab_key]
    for i, el in enumerate(vocab):
        if el == "None":
            vocab[i] = "~"
for vocab_key in ("s2i", "o2i"):
    tucker.vocab[vocab_key]["~"] = tucker.vocab[vocab_key].pop("None")
print(
    "None" in tucker.vocab["vocab_s"],
    "None" in tucker.vocab["vocab_o"],
    "None" in tucker.vocab["s2i"].keys(),
    "None" in tucker.vocab["o2i"].keys()
)

False False False False


In [44]:
from tucker_tensor import _role_index, _voc_index
from collections import defaultdict
import numpy as np
from tqdm import tqdm

def extend_vocab(role, sample, tucker):
    """
    Build representations for OOV tokens in the given role ("verb", "subject", "object")
    by averaging tucker.predicted_role_vector(...) across all contexts where the other
    two roles are in-vocab.

    Parameters
    ----------
    role : {"verb","subject","object"}
    sample : iterable of (verb, subject, object) triples (strings)
    tucker : TuckerDecomposition

    Returns
    -------
    extension : dict[str, np.ndarray]
        Mapping from OOV token -> averaged predicted role vector.
    out_contexts : dict[str, list[tuple]]
        Mapping from OOV token -> contexts used (optional use/debug).
    """
    if role not in {"verb", "subject", "object"}:
        raise ValueError(f"role must be one of {{'verb','subject','object'}}, got {role!r}")

    # indices in the (v,s,o) triple
    r_idx = _role_index(role)
    other_idxs = [i for i in (0, 1, 2) if i != r_idx]

    # vocab keys inside tucker.vocab
    this_vocab_key = _voc_index(role)
    other_vocab_keys = [_voc_index("verb"), _voc_index("subject"), _voc_index("object")]
    other_vocab_keys = [other_vocab_keys[i] for i in other_idxs]

    this_vocab = tucker.vocab[this_vocab_key]
    other_vocabs = [tucker.vocab[k] for k in other_vocab_keys]

    # collect contexts where element in `role` is OOV but the other two are in-vocab
    out = defaultdict(list)
    for vector in tqdm(sample, desc="building OOV add list"):
        element = vector[r_idx]

        # already in vocab => skip
        if element in this_vocab:
            continue

        # other roles must be in vocab
        if all(vector[o_i] in o_v for o_i, o_v in zip(other_idxs, other_vocabs)):
            out[element].append(vector)


    # compute mean predicted role vector per OOV element
    extension = {}
    for element, vectors in tqdm(out.items(), desc="calculating representations"):
        reps = [tucker.predicted_role_vector(vector, role) for vector in vectors]
        extension[element] = np.mean(reps, axis=0)

    return extension

def _contexts_for_oov(role, sample, vocabs_by_role):
    """
    Collect contexts where token in `role` is OOV but the other two roles are in-vocab.

    Parameters
    ----------
    role : {"verb","subject","object"}
    sample : iterable[(v,s,o)]
    vocabs_by_role : dict[str, set[str]]  # role -> current vocab set

    Returns
    -------
    out_contexts : dict[str, list[tuple]]
        OOV token -> list of contexts (triples) usable for imputing it.
    """
    r_idx = _role_index(role)
    other_roles = [r for r in ("verb", "subject", "object") if r != role]
    other_idxs = [_role_index(r) for r in other_roles]

    this_vocab = vocabs_by_role[role]
    other_vocabs = [vocabs_by_role[r] for r in other_roles]

    out = defaultdict(list)
    for triple in sample:
        tok = triple[r_idx]
        if tok in this_vocab:
            continue
        if all(triple[i] in v for i, v in zip(other_idxs, other_vocabs)):
            out[tok].append(triple)
    return out


def simulate_extend_vocab(role, sample, tucker, vocabs_by_role=None):
    """
    Simulation-only: returns which tokens *could* be added + operation counts,
    without calling tucker.predicted_role_vector.

    Returns
    -------
    tokens_to_add : set[str]
    out_contexts : dict[str, list[tuple]]
    n_ops : int  # how many predicted_role_vector calls would be needed
    """
    if role not in {"verb", "subject", "object"}:
        raise ValueError(f"role must be one of {{'verb','subject','object'}}, got {role!r}")

    if vocabs_by_role is None:
        # default: derive from tucker
        vocabs_by_role = {
            "verb": set(tucker.vocab[_voc_index("verb")]),
            "subject": set(tucker.vocab[_voc_index("subject")]),
            "object": set(tucker.vocab[_voc_index("object")]),
        }

    out_contexts = _contexts_for_oov(role, sample, vocabs_by_role)
    tokens_to_add = set(out_contexts.keys())
    n_ops = sum(len(ctxs) for ctxs in out_contexts.values())  # 1 op per context
    return tokens_to_add, out_contexts, n_ops


def simulate_bootstrap(sample, tucker, max_iters=50, roles=("verb", "subject", "object"), verbose=True):
    """
    Simulate iterative bootstrapping until no new tokens can be added.

    Returns
    -------
    history : list[dict]
        Per-iteration stats + token lists.
    final_vocabs : dict[str, set[str]]
        Vocab sets after simulation.
    """
    vocabs = {
        "verb": set(tucker.vocab[_voc_index("verb")]),
        "subject": set(tucker.vocab[_voc_index("subject")]),
        "object": set(tucker.vocab[_voc_index("object")]),
    }

    history = []
    for it in range(1, max_iters + 1):
        iter_add = {}
        iter_ops = {}
        iter_contexts = {}

        # compute addable tokens per role *given current vocabs*
        for role in roles:
            tokens_to_add, out_contexts, n_ops = simulate_extend_vocab(
                role=role, sample=sample, tucker=tucker, vocabs_by_role=vocabs
            )
            # important: don't "add" tokens already in vocab (shouldn't happen, but safe)
            tokens_to_add = tokens_to_add - vocabs[role]

            iter_add[role] = tokens_to_add
            iter_ops[role] = n_ops
            iter_contexts[role] = out_contexts

        total_new = sum(len(iter_add[r]) for r in roles)
        total_ops = sum(iter_ops[r] for r in roles)

        history.append({
            "iter": it,
            "new_verb": len(iter_add["verb"]),
            "new_subject": len(iter_add["subject"]),
            "new_object": len(iter_add["object"]),
            "new_total": total_new,
            "ops_verb": iter_ops["verb"],
            "ops_subject": iter_ops["subject"],
            "ops_object": iter_ops["object"],
            "ops_total": total_ops,
            "added_tokens": {r: sorted(iter_add[r]) for r in roles},
        })

        if verbose:
            print(
                f"[sim iter {it}] new: "
                f"v={len(iter_add['verb'])}, s={len(iter_add['subject'])}, o={len(iter_add['object'])} "
                f"(total={total_new}) | ops={total_ops}"
            )

        if total_new == 0:
            break

        # "commit" additions (simultaneous update per iteration)
        for role in roles:
            vocabs[role].update(iter_add[role])

    return history, vocabs



In [45]:
# Example usage:
# - if you already have sentence_sample, use it
# - otherwise swap in whatever sample loader you prefer
sim_history, sim_final_vocabs = simulate_bootstrap(sentence_sample, tucker, max_iters=50, verbose=True)

# quick peek: first few iterations
for h in sim_history[:3]:
    print(h["iter"], h["new_total"], h["ops_total"])


[sim iter 1] new: v=17681, s=211977, o=98466 (total=328124) | ops=2112421
[sim iter 2] new: v=6401, s=31068, o=24249 (total=61718) | ops=67933
[sim iter 3] new: v=233, s=739, o=618 (total=1590) | ops=1617
[sim iter 4] new: v=16, s=41, o=38 (total=95) | ops=98
[sim iter 5] new: v=3, s=10, o=8 (total=21) | ops=21
[sim iter 6] new: v=1, s=2, o=3 (total=6) | ops=6
[sim iter 7] new: v=0, s=0, o=2 (total=2) | ops=2
[sim iter 8] new: v=0, s=0, o=0 (total=0) | ops=0
1 328124 2112421
2 61718 67933
3 1590 1617


In [46]:
# Print a compact summary of added tokens (warning: can be long)
for h in sim_history:
    if h["new_total"] == 0:
        continue
    print(f"\n=== iter {h['iter']} ===")
    for role in ("verb", "subject", "object"):
        toks = h["added_tokens"][role]
        if toks:
            print(f"{role}: {len(toks)}")
            print("  ", ", ".join(toks[:50]), ("..." if len(toks) > 50 else ""))



=== iter 1 ===
verb: 17681
   ""those, "(47, "(6, ",through, "--, "--Dr, "--answere, "--provide, "-A, ".(i, "14, "2, "23, "25, "49, "6, "Click, "calle, "goe, "is, "jacke, "jennifer, "love, "please, "read, "shut, "tell, "visit, "we're, ', ''、ha, 'cause, 'follow, 'refresh, (:, *, +, +0, +19127292266, +2, +406, +447463151997, +49(0)711, +79788502838, +91, -, ---include, --adapte, --by, --provide ...
subject: 211977
   !, ", ""I'll, ""i, "(18, ","confirm_email_sent_message":"Please, ","give_agree_to_terms":"You, ","give_email":"please, ","give_first":"please, ","give_last":"Please, "-, "-Ruth, "16, "24, "26, "29, "3, "4, "5, "A, "China, "Chris, "Do, "Doug, "Dustin, "God, "Harry, "Hayden, "He, "Heat, "Hezbollah, "Historians, "Horejs, "Hylota, "I, "It, "Jackson, "Jesus, "Klaus, "Klopp, "Koalafi, "Kristen, "Ma, "Majority, "Mason, "NEMESIS, "Newsarama, "O'Connor, "Oh, "Ragnor ...
object: 98466
   !, ", "7, "bastard, "blue, "just, "what, #, $, $71.19, &, ', ''”14, 'S, 's, 've, *, +, +0.343, +1

In [47]:
verb_extension = extend_vocab("verb", sentence_sample, tucker)

calculating representations:   0%|          | 55/17681 [00:09<51:23,  5.72it/s]  


KeyboardInterrupt: 

In [48]:
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import numpy as np
from tqdm import tqdm
from similarity import get_eval_num_threads
from utils import ThreadBudget  # same as your eval code
# reuse your helper from the repo
# from <wherever> import get_eval_num_threads


def extend_vocab(
    role,
    sample,
    tucker,
    *,
    n_threads: int | None = None,
    thread_budget: ThreadBudget | None = None,
    fraction_threads: float = 0.75,
    min_threads: int = 1,
):
    """
    Build representations for OOV tokens in the given role ("verb", "subject", "object")
    by averaging tucker.predicted_role_vector(...) across all contexts where the other
    two roles are in-vocab.

    Threaded:
      - parallelizes predicted_role_vector calls with a ThreadPool
      - uses repo-style get_eval_num_threads() by default
      - optionally respects a ThreadBudget limiter (same pattern as evaluate_sample)
    """
    if role not in {"verb", "subject", "object"}:
        raise ValueError(f"role must be one of {{'verb','subject','object'}}, got {role!r}")

    # repo-style default threads
    if n_threads is None:
        n_threads = get_eval_num_threads(fraction=fraction_threads, min_threads=min_threads)

    # indices in the (v,s,o) triple
    r_idx = _role_index(role)
    other_idxs = [i for i in (0, 1, 2) if i != r_idx]

    # vocab keys inside tucker.vocab
    this_vocab_key = _voc_index(role)
    other_vocab_keys = [_voc_index("verb"), _voc_index("subject"), _voc_index("object")]
    other_vocab_keys = [other_vocab_keys[i] for i in other_idxs]

    this_vocab = tucker.vocab[this_vocab_key]
    other_vocabs = [tucker.vocab[k] for k in other_vocab_keys]

    # collect contexts where element in `role` is OOV but the other two are in-vocab
    out = defaultdict(list)
    for vector in tqdm(sample, desc="building OOV add list"):
        element = vector[r_idx]

        # already in vocab => skip
        if element in this_vocab:
            continue

        # other roles must be in vocab
        if all(vector[o_i] in o_v for o_i, o_v in zip(other_idxs, other_vocabs)):
            out[element].append(vector)

    if not out:
        return {}

    # ThreadBudget limiter (same pattern as evaluate_sample)
    limiter = (thread_budget.limit() if thread_budget is not None else None)
    if limiter is None:
        ctx = contextmanager(lambda: (yield))()  # no-op
    else:
        ctx = limiter

    # We accumulate sums+counts per token to avoid storing all reps in RAM
    sums: dict[str, np.ndarray] = {}
    counts: dict[str, int] = {}

    def _one_call(tok, ctx_triple):
        rep = tucker.predicted_role_vector(ctx_triple, role)
        return tok, np.asarray(rep)

    # flatten jobs: 1 job per usable context
    jobs = [(tok, ctx_triple) for tok, ctxs in out.items() for ctx_triple in ctxs]

    with ctx:
        with ThreadPoolExecutor(max_workers=n_threads) as ex:
            futures = [ex.submit(_one_call, tok, ctx_triple) for tok, ctx_triple in jobs]

            for fut in tqdm(as_completed(futures), total=len(futures), desc="calculating representations"):
                tok, rep = fut.result()

                if tok not in sums:
                    sums[tok] = rep.copy()
                    counts[tok] = 1
                else:
                    sums[tok] += rep
                    counts[tok] += 1

    extension = {tok: sums[tok] / counts[tok] for tok in sums.keys()}
    return extension


In [49]:
n_threads = get_eval_num_threads(0.33)
verb_extension = extend_vocab("verb", sentence_sample, tucker, n_threads=n_threads)


calculating representations: 100%|██████████| 140599/140599 [00:24<00:00, 5757.87it/s]


In [50]:
verb_extension

{'hone': array([1.82909395e+04, 2.72589258e+04, 6.44821680e+04, 9.16012344e+04,
        5.48665430e+04, 2.54057520e+04, 4.14727695e+04, 5.63453906e+04,
        1.83276426e+04, 1.64970984e+05, 6.62783203e+04, 5.68386719e+04,
        3.80479609e+04, 3.16374355e+04, 1.57173344e+05, 2.04088613e+04,
        3.89818008e+04, 2.99650156e+04, 3.50622266e+04, 1.31910906e+05,
        7.86252422e+04, 3.45471797e+04, 3.77982695e+04, 4.10929883e+04,
        1.69547797e+05, 6.98059438e+05, 4.98055703e+04, 2.98974609e+04,
        5.96342500e+04, 6.44938867e+04, 1.18666094e+05, 3.77920312e+04,
        9.23315156e+04, 1.14434125e+05, 2.64300996e+04, 3.85565352e+04,
        2.96314062e+05, 9.36645781e+04, 1.25363359e+05, 4.19259688e+04,
        6.08552539e+04, 3.44672109e+04, 1.27917219e+05, 8.38462266e+04,
        7.28793906e+04, 5.28990547e+04, 3.61445859e+04, 3.04832305e+04,
        3.66799531e+04, 1.59006375e+05, 1.32854062e+05, 1.27788984e+05,
        1.72517234e+05, 6.52047461e+04, 3.74532773e+04, 

In [51]:
len(verb_extension)

17681